# Notebook 03: Modelado de Machine Learning, Optimización y Evaluación
Este notebook entrena los dos clasificadores definidos para predecir la fuga:
1. **Baseline:** Regresión Logística con balance de clases.
2. **Modelo Principal:** XGBoost optimizado mediante una búsqueda por rejilla (`GridSearchCV`).

Finalmente, realiza la evaluación final rigurosa en el conjunto de prueba independiente (`test set`) para medir el rendimiento de generalización sin sesgos.

In [ ]:
import os
import sys
import pandas as pd
import joblib

# Asegurar que podemos importar desde src
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from src.models.model_pipeline import train_models
from src.evaluation.metrics_eval import evaluate_on_test

print("Módulos importados correctamente.")

## 1. Cargar Datos Procesados y Preprocesador

In [ ]:
# Cargar datos
train_df = pd.read_csv("../data/processed/train.csv")
val_df = pd.read_csv("../data/processed/val.csv")
test_df = pd.read_csv("../data/processed/test.csv")

# Cargar preprocesador entrenado
preprocesador = joblib.load("../models/checkpoints/preprocessor.joblib")

# Separar características y variable objetivo
X_train, y_train = train_df.drop(columns=["Churn"]), train_df["Churn"]
X_val, y_val = val_df.drop(columns=["Churn"]), val_df["Churn"]
X_test, y_test = test_df.drop(columns=["Churn"]), test_df["Churn"]

print(f"Registros cargados: Train: {X_train.shape[0]} | Val: {X_val.shape[0]} | Test: {X_test.shape[0]}")

## 2. Entrenamiento y Optimización de Modelos
Llamamos al módulo `train_models` que entrena el baseline e implementa una búsqueda de cuadrícula (`GridSearchCV`) sobre XGBoost, seleccionando los mejores hiperparámetros que optimizan el F1-Score en validación. Posteriormente, exporta los modelos resultantes.

In [ ]:
# Entrenar modelos y guardar checkpoints
best_xgb, baseline_lr = train_models(
    X_train, 
    y_train, 
    X_val, 
    y_val, 
    preprocesador, 
    checkpoints_dir="../models/checkpoints"
)

## 3. Evaluación Final Rigurosa en el Conjunto de Prueba
Utilizando el conjunto de prueba independiente (`X_test`, `y_test`), evaluamos el rendimiento final de generalización del clasificador XGBoost optimizado. Esto evitará sesgos y falsas expectativas de rendimiento basadas únicamente en el conjunto de validación.

In [ ]:
# Evaluar y generar matriz de confusión en el conjunto de prueba
test_metrics = evaluate_on_test(
    best_xgb, 
    preprocesador, 
    X_test, 
    y_test,
    output_dir="../outputs"
)